In [33]:
import pandas as pd
import os
import numpy as np
from tqdm import tqdm
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import random 

In [53]:
def mask_to_yolo_labels(mask, output_txt_path = None, value_to_class_mapping = None, min_area=10, save = False):
    """
    將包含多個結節的 mask 轉換成 YOLO 格式的 labels.txt
    支援多值 mask（如 0:background, 1:nodule, 2:maybe_nodule）
    
    Args:
        mask_path: mask 圖片的路徑
        output_txt_path: 輸出的 labels.txt 路徑
        value_to_class_mapping: dict，將 mask 值映射到 YOLO 類別 ID
                               例如：{1: 0, 2: 1} 表示 mask 值 1 -> class 0, mask 值 2 -> class 1
                               如果為 None，則將所有非零值視為 class 0
        min_area: 最小區域面積，用於過濾過小的噪點（預設為 10 像素）
    
    Example:
        # 將 mask 值 1 標註為 class 0 (nodule)，mask 值 2 標註為 class 1 (maybe_nodule)
        mask_to_yolo_labels(
            mask_path="sample.png",
            output_txt_path="sample.txt",
            value_to_class_mapping={1: 0, 2: 1}
        )
    """
    
    # 取得圖片尺寸
    height, width = mask.shape
    
    # 如果沒有指定映射，預設將所有非零值視為 class 0
    if value_to_class_mapping is None:
        value_to_class_mapping = {1: 0}
    
    # 準備寫入的標註列表
    yolo_labels = []
    
    # 對每個 mask 值分別處理
    for mask_value, class_id in value_to_class_mapping.items():
        # 創建該值的二值 mask
        binary_mask = (mask == mask_value).astype(np.uint8) * 255
        
        # 使用連通組件分析找出所有獨立的結節
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
        
        # 遍歷每個連通組件（跳過背景，即 label 0）
        for i in range(1, num_labels):
            # 取得邊界框信息
            x = stats[i, cv2.CC_STAT_LEFT]
            y = stats[i, cv2.CC_STAT_TOP]
            w = stats[i, cv2.CC_STAT_WIDTH]
            h = stats[i, cv2.CC_STAT_HEIGHT]
            area = stats[i, cv2.CC_STAT_AREA]
            
            # 過濾掉過小的區域（可能是噪點）
            if area < min_area:
                continue
            
            # 計算 YOLO 格式的歸一化坐標
            # YOLO 格式: class_id x_center y_center width height（都是歸一化到 0-1）
            x_center = (x + w / 2) / width
            y_center = (y + h / 2) / height
            norm_width = w / width
            norm_height = h / height
            
            # 添加到標註列表
            yolo_labels.append(f"{class_id} {x_center:.6f} {y_center:.6f} {norm_width:.6f} {norm_height:.6f}")
    
    # 寫入文件
    if save:
        with open(output_txt_path, 'w') as f:
            f.write('\n'.join(yolo_labels))
        
        print(f"Found {len(yolo_labels)} nodules in {mask_path}")
        print(f"Saved to {output_txt_path}")
        if len(yolo_labels) == 0:
            print("Warning: No nodules found in the mask. Check the mask values and mapping.")
            # print(error)
    
    return len(yolo_labels)

# Prepare TN3K data

In [4]:
def get_TN3K_df(data_root):
    """從資料夾中讀取圖片檔名，並建立一個 DataFrame 包含 patient_ID、file_name、nodule_type、train_or_test"""
    data = {
        "dataset": [],
        "patient_ID": [],
        "file_name": [],
        "nodule_type": [],
        "train_or_test": []
    }

    for patient_ID in tqdm(os.listdir(data_root)):
        if "DS_Store" in patient_ID:
            continue
        for file_name in os.listdir(f"{data_root}/{patient_ID}/gray_image"):
            if file_name.endswith(".jpg") or file_name.endswith(".png"):
                file_name = file_name.replace(".jpg", "")
                if "train" in file_name:
                    train_or_test = "train"
                elif "test" in file_name:
                    train_or_test = "test"
                mask = Image.open(f"{data_root}/{patient_ID}/mask/{file_name}.jpg").convert("L")
                mask = np.array(mask)
                num_nodule = mask_to_yolo_labels(mask)
                if num_nodule == 0:
                    nodule_type = "none"
                elif num_nodule == 1:
                    nodule_type = "single"
                else:
                    nodule_type = "multiple"
                data["dataset"].append("TN3K")
                data["patient_ID"].append(patient_ID)
                data["file_name"].append(file_name)
                data["nodule_type"].append(nodule_type)
                data["train_or_test"].append(train_or_test)
    df = pd.DataFrame(data)
    return df

In [5]:
data_root = "../data/TN3K/all_data_png/nodule"
TN3K_df = get_TN3K_df(data_root)

100%|██████████| 2/2 [00:01<00:00,  1.55it/s]


In [6]:
TN3K_df

,dataset,patient_ID,file_name,nodule_type,train_or_test
0,TN3K,NO,test_0191,single,test
1,TN3K,NO,train_1922,single,train
2,TN3K,NO,train_0382,single,train
3,TN3K,NO,train_2595,single,train
4,TN3K,NO,train_2581,single,train
...,...,...,...,...,...
3488,TN3K,NO,train_2598,single,train
3489,TN3K,NO,train_1085,single,train
3490,TN3K,NO,test_0188,multiple,test
3491,TN3K,NO,train_1913,single,train


# Prepare TN5000 data

In [7]:
def get_TN5000_df(data_root, train_file_name, val_file_name, test_file_name):
    """從資料夾中讀取圖片檔名，並建立一個 DataFrame 包含 patient_ID、file_name、nodule_type、train_or_test"""
    data = {
        "dataset": [],
        "patient_ID": [],
        "file_name": [],
        "nodule_type": [],
        "train_or_test": []
    }

    for patient_ID in tqdm(os.listdir(data_root)):
        if "DS_Store" in patient_ID:
            continue
        for file_name in os.listdir(f"{data_root}/{patient_ID}/gray_image"):
            if file_name.endswith(".jpg") or file_name.endswith(".png"):
                file_name = file_name.replace(".jpg", "")
                if file_name in train_file_name:
                    train_or_test = "train"
                elif file_name in val_file_name:
                    train_or_test = "test"
                elif file_name in test_file_name:
                    train_or_test = "test"
                else:
                    print(f"Warning: {file_name} not found in train, val, or test ID lists.")
                    continue
                
                mask = Image.open(f"{data_root}/{patient_ID}/mask/{file_name}.png").convert("L")
                mask = np.array(mask)
                num_nodule = mask_to_yolo_labels(mask)
                if num_nodule == 0:
                    nodule_type = "none"
                elif num_nodule == 1:
                    nodule_type = "single"
                else:
                    nodule_type = "multiple"
                data["dataset"].append("TN5000")
                data["patient_ID"].append(patient_ID)
                data["file_name"].append(file_name)
                data["nodule_type"].append(nodule_type)
                data["train_or_test"].append(train_or_test)
    df = pd.DataFrame(data)
    return df

In [8]:
data_root = "../data/TN5000/all_data_png/nodule"
with open("../data/TN5000/train.txt", "r") as f:
    train_ID = f.read().splitlines()
with open("../data/TN5000/val.txt", "r") as f:
    val_ID = f.read().splitlines()
with open("../data/TN5000/test.txt", "r") as f:
    test_ID = f.read().splitlines()
TN5000_df = get_TN5000_df(data_root, train_ID, val_ID, test_ID)

100%|██████████| 2/2 [00:04<00:00,  2.12s/it]


In [9]:
TN5000_df

,dataset,patient_ID,file_name,nodule_type,train_or_test
0,TN5000,NO,003301,single,train
1,TN5000,NO,001516,single,train
2,TN5000,NO,000608,single,train
3,TN5000,NO,001270,single,test
4,TN5000,NO,002779,single,train
...,...,...,...,...,...
4995,TN5000,NO,000177,single,test
4996,TN5000,NO,000611,single,train
4997,TN5000,NO,002006,single,train
4998,TN5000,NO,003318,single,train


# Prepare CG data

In [40]:
def get_CG_df_nodule(data_root):
    """從資料夾中讀取圖片檔名，並建立一個 DataFrame 包含 patient_ID、file_name、nodule_type、train_or_test"""
    data = {
        "dataset": [],
        "patient_ID": [],
        "file_name": [],
        "nodule_type": [],
        "train_or_test": []
    }

    for patient_ID in tqdm(os.listdir(data_root)):
        if "DS_Store" in patient_ID:
            continue
        if random.random() < 0.8:
            train_or_test = "train"
        else:            
            train_or_test = "test"
            
        for file_name in os.listdir(f"{data_root}/{patient_ID}/gray_image"):
            if file_name.endswith(".jpg") or file_name.endswith(".png"):
                file_name = file_name.replace(".png", "")
                mask = Image.open(f"{data_root}/{patient_ID}/mask/{file_name}.png").convert("L")
                mask = np.array(mask)
                num_nodule = mask_to_yolo_labels(mask)
                if num_nodule == 0:
                    nodule_type = "none"
                    print(f"No nodules found in {file_name} for patient {patient_ID}")
                elif num_nodule == 1:
                    nodule_type = "single"
                else:
                    nodule_type = "multiple"
                data["dataset"].append("CG")
                data["patient_ID"].append(patient_ID)
                data["file_name"].append(file_name)
                data["nodule_type"].append(nodule_type)
                data["train_or_test"].append(train_or_test)
    df = pd.DataFrame(data)
    return df

In [41]:
def get_CG_df_normal(data_root):
    """從資料夾中讀取圖片檔名，並建立一個 DataFrame 包含 patient_ID、file_name、nodule_type、train_or_test"""
    data = {
        "dataset": [],
        "patient_ID": [],
        "file_name": [],
        "nodule_type": [],
        "train_or_test": []
    }

    for patient_ID in tqdm(os.listdir(data_root)):
        if "DS_Store" in patient_ID:
            continue
        if random.random() < 0.8:
            train_or_test = "train"
        else:            
            train_or_test = "test"
            
        for file_name in os.listdir(f"{data_root}/{patient_ID}/gray_image"):
            if file_name.endswith(".jpg") or file_name.endswith(".png"):
                file_name = file_name.replace(".png", "")
                nodule_type = "none"
                data["dataset"].append("CG")
                data["patient_ID"].append(patient_ID)
                data["file_name"].append(file_name)
                data["nodule_type"].append(nodule_type)
                data["train_or_test"].append(train_or_test)
    df = pd.DataFrame(data)
    return df

In [42]:
nodule_data_root = "../data/CG_data/all_data_png/nodule_clean"
CG_df_nodule = get_CG_df_nodule(nodule_data_root)

normal_data_root = "../data/CG_data/all_data_png/normal_clean"
CG_df_normal = get_CG_df_normal(normal_data_root)

100%|██████████| 1129/1129 [00:00<00:00, 83469.10it/s]


In [43]:
train_CG_df_nodule = CG_df_nodule[CG_df_nodule["train_or_test"] == "train"]
test_CG_df_nodule = CG_df_nodule[CG_df_nodule["train_or_test"] == "test"]
print("train ID lens : ", len(train_CG_df_nodule["patient_ID"].unique()))
print("test ID lens : ", len(test_CG_df_nodule["patient_ID"].unique()))

train ID lens :  1658
test ID lens :  406


In [44]:
train_CG_df_normal = CG_df_normal[CG_df_normal["train_or_test"] == "train"]
test_CG_df_normal = CG_df_normal[CG_df_normal["train_or_test"] == "test"]
print("train ID lens : ", len(train_CG_df_normal["patient_ID"].unique()))
print("test ID lens : ", len(test_CG_df_normal["patient_ID"].unique()))

train ID lens :  917
test ID lens :  211


In [45]:
CG_df_nodule["train_or_test"].value_counts()

train_or_test
train    5899
test     1444
Name: count, dtype: int64

In [46]:
CG_df_normal["train_or_test"].value_counts()

train_or_test
train    3889
test      920
Name: count, dtype: int64

In [47]:
combine_df = pd.concat([TN3K_df, TN5000_df, CG_df_nodule, CG_df_normal], ignore_index=True)

In [48]:
combine_df.to_csv("../yolo_data/all_csv_files/combined_dataset_info_v5.csv", index=False)

In [56]:
import shutil
def copy_file_to_yolo_data(combine_df):
    for idx, row in tqdm(combine_df.iterrows(), total=combine_df.shape[0]):
        dataset = row["dataset"]
        patient_ID = row["patient_ID"]
        file_name = row["file_name"]
        train_or_test = row["train_or_test"]
        nodule_type = row["nodule_type"]
        
        if dataset == "TN3K":
            src_image_path = f"../data/TN3K/all_data_png/nodule/{patient_ID}/gray_image/{file_name}.jpg"
            src_mask_path = f"../data/TN3K/all_data_png/nodule/{patient_ID}/mask/{file_name}.jpg"
        elif dataset == "TN5000":
            src_image_path = f"../data/TN5000/all_data_png/nodule/{patient_ID}/gray_image/{file_name}.jpg"
            src_mask_path = f"../data/TN5000/all_data_png/nodule/{patient_ID}/mask/{file_name}.png"
        elif dataset == "CG":
            if nodule_type != "none":
                src_image_path = f"../data/CG_data/all_data_png/nodule_clean/{patient_ID}/denoise_cut/{file_name}.png"
                src_mask_path = f"../data/CG_data/all_data_png/nodule_clean/{patient_ID}/mask_cut/{file_name}.png"
            else:
                src_image_path = f"../data/CG_data/all_data_png/normal_clean/{patient_ID}/denoise_cut/{file_name}.png"
                src_mask_path = f"../data/CG_data/all_data_png/normal_clean/{patient_ID}/mask_cut/{file_name}.png"
        else:
            print(f"Unknown dataset: {dataset}")
            continue
        shutil.copy(src_image_path, f"../yolo_data/all_data_nodule_normal_cut/images/{file_name}.png")
        shutil.copy(src_mask_path, f"../yolo_data/all_data_nodule_normal_cut/masks/{file_name}.png")
        

In [59]:
copy_file_to_yolo_data(combine_df)

100%|██████████| 20645/20645 [00:11<00:00, 1851.42it/s]


In [55]:
print(len(os.listdir("../yolo_data/all_data_nodule_cut/images")))
print(len(os.listdir("../yolo_data/all_data_nodule_cut/masks")))
print(len(os.listdir("../yolo_data/all_data_nodule_cut/labels")))

20645
20645
20645


# mask to labels

In [60]:
data_root = "../yolo_data/all_data_nodule_normal_cut/masks"
for mask_name in tqdm(os.listdir(data_root)):
    if mask_name.endswith(".png"):
        mask_path = f"{data_root}/{mask_name}"
        output_txt_path = f"../yolo_data/all_data_nodule_normal_cut/labels/{mask_name.replace('.png', '.txt')}"
        mask = Image.open(mask_path).convert("L")
        mask = np.array(mask)
        mask_to_yolo_labels(mask, output_txt_path, value_to_class_mapping={1: 0, 2: 1}, save=True)

  1%|          | 172/20645 [00:00<00:23, 889.23it/s]

Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/ba15d09e317a7e35a3ced7908a7599fc_right.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/ba15d09e317a7e35a3ced7908a7599fc_right.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/a9e8df661a05a4363f80f08b28aada02.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/a9e8df661a05a4363f80f08b28aada02.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/91390c89aa459cb587f2077e2b5e6461.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/91390c89aa459cb587f2077e2b5e6461.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/004863.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/004863.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/test_0378.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/test_0378.txt
Found 2 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/b94f871586a4403ca5131da58bca6d33.png
Saved to 

  1%|▏         | 262/20645 [00:00<00:23, 849.58it/s]

Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/train_0369.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/train_0369.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/49306ecb5be017d58950403844080e6d_right.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/49306ecb5be017d58950403844080e6d_right.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/002976.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/002976.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/train_1077.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/train_1077.txt
Found 3 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/train_1711.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/train_1711.txt
Found 0 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/331c09c546e28273c45aa9c4cf5f7382_right.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/331c09c546e28273c45aa9c4cf5f7382_

  3%|▎         | 533/20645 [00:00<00:17, 1159.88it/s]

...

Found 0 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/f5391412cedb1f8b109be468a0fe84de_right.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/f5391412cedb1f8b109be468a0fe84de_right.txt
Found 0 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/bf03ce6fef6b4bbc7b7e5efc67ba5feb.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/bf03ce6fef6b4bbc7b7e5efc67ba5feb.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/001678.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/001678.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/e70fc6d944990297af54b62dc90112f5_right.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/e70fc6d944990297af54b62dc90112f5_right.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/002371.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/002371.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/004700.png
Saved to ../yolo_data/all_dat

 99%|█████████▊| 20382/20645 [00:17<00:00, 1238.02it/s]

Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/000995.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/000995.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/001453.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/001453.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/003244.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/003244.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/train_1683.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/train_1683.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/003522.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/003522.txt
Found 0 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/95d0bba0b07618bef98411fc1a8c9a91_left.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/95d0bba0b07618bef98411fc1a8c9a91_left.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/dd28ceb

100%|██████████| 20645/20645 [00:17<00:00, 1163.43it/s]

Found 2 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/train_0416.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/train_0416.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/train_1708.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/train_1708.txt
Found 0 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/d06ed041c5306a3698c5918b20532e21_right.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/d06ed041c5306a3698c5918b20532e21_right.txt
Found 0 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/2cdefe0cdd54a3f37bd4c39d5a2b146a_right.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/2cdefe0cdd54a3f37bd4c39d5a2b146a_right.txt
Found 1 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/65cad74012e85e768ff8c2be7ae9f9f2.png
Saved to ../yolo_data/all_data_nodule_normal_cut/labels/65cad74012e85e768ff8c2be7ae9f9f2.txt
Found 2 nodules in ../yolo_data/all_data_nodule_normal_cut/masks/test_0163.png
Saved to .

In [61]:
import matplotlib.pyplot as plt

def visualize_yolo_labels(image_path, label_path, class_names=None, save_path=None):
    """
    可視化 YOLO 標註，在原圖上繪製邊界框
    支援多類別顯示（不同類別用不同顏色）
    
    Args:
        image_path: 原始圖片路徑
        label_path: YOLO labels.txt 路徑
        class_names: dict，類別 ID 到名稱的映射，例如 {0: "nodule", 1: "maybe_nodule"}
        save_path: 保存可視化結果的路徑（可選）
    """
    # 讀取圖片
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Cannot read image from {image_path}")
        return
    
    height, width = img.shape[:2]
    
    # 讀取標註
    if not os.path.exists(label_path):
        print(f"Error: Label file not found at {label_path}")
        return
    
    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    # 定義不同類別的顏色
    colors = [
        (0, 255, 0),    # Green - class 0
        (255, 0, 0),    # Blue - class 1
        (0, 165, 255),  # Orange - class 2
        (255, 0, 255),  # Magenta - class 3
    ]
    
    if class_names is None:
        class_names = {0: "nodule", 1: "maybe_nodule"}
    
    # 繪製每個邊界框
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        
        class_id = int(float(parts[0]))
        x_center, y_center, w, h = map(float, parts[1:5])
        
        # 將歸一化坐標轉換回像素坐標
        x_center_px = x_center * width
        y_center_px = y_center * height
        w_px = w * width
        h_px = h * height
        
        # 計算左上角和右下角坐標
        x1 = int(x_center_px - w_px / 2)
        y1 = int(y_center_px - h_px / 2)
        x2 = int(x_center_px + w_px / 2)
        y2 = int(y_center_px + h_px / 2)
        
        # 選擇顏色
        color = colors[class_id % len(colors)]
        
        # 繪製邊界框
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        
        # 標註類別名稱
        label = class_names.get(class_id, f"Class {class_id}")
        cv2.putText(img, label, (x1, y1-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    # 保存
    if save_path:
        cv2.imwrite(save_path, img)
        print(f"Visualization saved to {save_path}")
    
    # 轉換為 RGB 以便在 notebook 中顯示
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img_rgb

file_name = "87233ca73b2cc5c2f86a35ecb2f776ac_right"
# 使用範例 1: 單一類別
img_with_boxes = visualize_yolo_labels(
    image_path=f"../yolo_data/all_data_nodule/images/{file_name}.png",
    label_path=f"../yolo_data/all_data_nodule/labels/{file_name}.txt",
    class_names={0: "nodule", 1: "maybe_nodule"},
    save_path="visualization.png"
)

# 使用範例 2: 多類別（綠色=確定結節，藍色=可能結節）
# img_with_boxes = visualize_yolo_labels(
#     image_path="path/to/original/image.png",
#     label_path="path/to/labels.txt",
#     class_names={0: "nodule", 1: "maybe_nodule"},
#     save_path="visualization_multi_class.png"
# )

# 顯示結果
# plt.figure(figsize=(12, 12))
# plt.imshow(img_with_boxes)
# plt.axis('off')
# plt.title('YOLO Labels Visualization')
# plt.show()

Visualization saved to visualization.png


# Make train/val txt

In [63]:
data_type = "all_data_nodule_normal_cut"

In [64]:
import pandas as pd
def get_txt(df):
    image_paths = []
    for idx, row in df.iterrows():
        file_name = row["file_name"]
        image_path = f"yolo_data/{data_type}/images/{file_name}.png"
        image_paths.append(image_path)
    return image_paths

In [65]:
data_version = "v5"
df = pd.read_csv(f"../yolo_data/all_csv_files/combined_dataset_info_{data_version}.csv")
train_df = df[df["train_or_test"] == "train"]
val_df = df[(df["train_or_test"] == "test")]
val_df_CG = df[(df["train_or_test"] == "test") & (df["dataset"] == "CG")]
val_df_CG_single = val_df_CG[val_df_CG["nodule_type"] == "single"]

In [66]:
output_root = f"../yolo_data/{data_type}"

In [67]:
train_txt_paths = get_txt(train_df)
with open(f'{output_root}/train_{data_version}.txt', 'w') as f:
    for path in train_txt_paths:
        f.write(f"{path}\n")

In [68]:
val_txt_paths = get_txt(val_df)
with open(f'{output_root}/val_{data_version}.txt', 'w') as f:
    for path in val_txt_paths:
        f.write(f"{path}\n")

In [69]:
val_txt_paths = get_txt(val_df_CG)
with open(f'{output_root}/val_{data_version}_CG.txt', 'w') as f:
    for path in val_txt_paths:
        f.write(f"{path}\n")

In [70]:
val_txt_paths = get_txt(val_df_CG_single)
with open(f'{output_root}/val_{data_version}_CG_single.txt', 'w') as f:
    for path in val_txt_paths:
        f.write(f"{path}\n")